This notebook goes with the main notebook GA_Neural_Network_Module05_iNaturalist.ipynb

The purpose is to use the iNaturalist API to create the dataset that will be used to train the neural network.

In [ ]:
import requests, pandas as pd

PLACE_ID = 2345          # Miami-Dade County
START = "2023-11-26"     # 3 years back
URL = "https://api.inaturalist.org/v1/observations"
HEADERS = {"User-Agent": "MDC-CAI2300C-student-project"}

params = {"place_id": PLACE_ID, "quality_grade": "research", "captive": "false",
          "d1": START, "per_page": 200, "order_by": "id", "order": "asc", "id_above": 0}
data = requests.get(URL, params=params, headers=HEADERS).json()

print("Total sightings:", data["total_results"])
first = data["results"][0]
print("Species:", first["taxon"]["name"], "/", first["taxon"].get("preferred_common_name"))
print("Group:  ", first["taxon"]["iconic_taxon_name"])
print("Where:  ", first["location"])
print("When:   ", first["time_observed_at"])

Total sightings: 238467
Species: Hemidactylus frenatus / Common House Gecko
Group:   Reptilia
Where:   25.81998,-80.2221366667
When:    2023-11-26T01:56:21-05:00


In [ ]:
import time

rows, last_id = [], 0

def pull_page(last_id):
    p = {**params, "id_above": last_id}
    for attempt in range(5):
        try:
            r = requests.get(URL, params=p, headers=HEADERS, timeout=60)
            if r.status_code == 200:
                return r.json()["results"]
        except requests.RequestException:
            pass
        time.sleep(5 * (attempt + 1))   # wait longer after each failure
    raise RuntimeError("API kept failing. Run the next cell again to resume.")

In [ ]:
start = time.time()
while True:
    page = pull_page(last_id)
    if not page:
        break
    for o in page:
        t = o.get("taxon") or {}
        rows.append({
            "species": t.get("name"),
            "common_name": t.get("preferred_common_name"),
            "group": t.get("iconic_taxon_name"),
            "rank": t.get("rank"),
            "location": o.get("location"),
            "observed_at": o.get("time_observed_at"),
            "observed_on": o.get("observed_on"),
        })
    last_id = page[-1]["id"]
    if len(rows) % 10000 < 200:
        print(f"{len(rows):,} rows · {time.time() - start:.0f}s")
    time.sleep(1)   # be polite: about 1 request per second
print(f"Done: {len(rows):,} rows")

10,000 rows · 169s
20,000 rows · 342s
30,000 rows · 515s
40,000 rows · 687s
50,000 rows · 864s
60,000 rows · 1042s
70,000 rows · 1207s
80,000 rows · 1373s
90,000 rows · 1542s
100,000 rows · 1709s
110,000 rows · 1874s
120,000 rows · 2038s
130,000 rows · 2204s
140,000 rows · 2362s
150,000 rows · 2518s
160,000 rows · 2689s
170,000 rows · 2852s
180,000 rows · 3015s
190,000 rows · 3187s
200,000 rows · 3354s
210,000 rows · 3526s
220,000 rows · 3689s
230,000 rows · 3848s
Done: 238,475 rows


In [ ]:
import os

In [ ]:
df = pd.DataFrame(rows)
print(f"Downloaded: {len(df):,} rows")

# 1. Keep sightings identified to species, with a location and an exact time
df = df[df["rank"] == "species"].dropna(subset=["location", "observed_at"])

# 2. Split "lat,lon" into two number columns
df[["latitude", "longitude"]] = df["location"].str.split(",", expand=True).astype(float)

# 3. Week of year and local hour (Miami time)
when = pd.to_datetime(df["observed_at"], utc=True).dt.tz_convert("America/New_York")
df["week"] = when.dt.isocalendar().week.astype(int)
df["hour"] = when.dt.hour

# 4. A readable name to predict: "Common House Gecko (Hemidactylus frenatus)"
df["name"] = df["common_name"].fillna(df["species"]) + " (" + df["species"] + ")"

# 5. Keep species with at least 50 sightings
counts = df["name"].value_counts()
df = df[df["name"].isin(counts[counts >= 200].index)]

out = df[["name", "latitude", "longitude", "week", "hour"]]
out.to_csv("inat-miami.csv", index=False)
print(f"Saved: {len(out):,} rows · {out['name'].nunique()} species")
print(f"File size: {os.path.getsize('inat-miami.csv') / 1e6:.1f} MB")
out.head()

Downloaded: 238,475 rows
Saved: 147,878 rows · 252 species
File size: 10.4 MB


,name,latitude,longitude,week,hour
2,Florida Carpenter Ant (Camponotus floridanus),25.574243,-80.400309,47,0
6,Black Spiny-tailed Iguana (Ctenosaura similis),25.734646,-80.159260,47,9
8,Great Barracuda (Sphyraena barracuda),25.746338,-80.173630,47,12
10,Checkered Puffer (Sphoeroides testudineus),25.746622,-80.173905,47,12
13,Royal Tern (Thalasseus maximus),25.707867,-80.151864,47,12
